# 02 – Coherence QA/QC

This notebook validates and visually inspects the UAVSAR coherence GeoTIFFs
produced by the `scripts/slurms/calc_coherence/calc_coherence.py` pipeline.

**Sections**
1. Imports & Setup
2. File Discovery & Sampling
3. Data Validation (reasonable coherence values)
4. Downsampled Preview Plots

## 1. Imports & Setup

In [ ]:
import random
from pathlib import Path

import numpy as np
import xarray as xr
import rioxarray  # noqa: F401 – registers the .rio accessor on xarray objects
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Global matplotlib defaults for readable, reasonably sized figures
# ------------------------------------------------------------------
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 9

## 2. File Discovery & Sampling

Set `COHERENCE_DIR` to the root directory that contains the coherence GeoTIFFs
generated by the pipeline (the value you used for `--out_dir` when running
`calc_coherence.py`).  The cell then discovers every `*coh.tif` file
recursively and splits them into two lists:

* **interferometric** – filenames that end with `int_coh.tif`
* **copol** – filenames that end with `copol_coh.tif`

A random subset of up to `SAMPLE_SIZE` files is drawn from each list to keep
the notebook fast on large datasets.

In [ ]:
# ---------------------------------------------------------------
# USER CONFIGURATION – update this path before running
# ---------------------------------------------------------------
COHERENCE_DIR = Path("/bsuhome/julialober/scratch/coherence_data/coherence")

# Number of random files to sample from each coherence type
SAMPLE_SIZE = 5

# Fixed random seed for reproducibility; set to None for a fresh random draw
RANDOM_SEED = 42

# ---------------------------------------------------------------
# Discovery – recursively find every *coh.tif produced by the pipeline
# ---------------------------------------------------------------
all_coh_files = sorted(COHERENCE_DIR.rglob("*coh.tif"))
print(f"Total coherence files found: {len(all_coh_files)}")

# Split by coherence type based on the filename suffix convention:
#   interferometric:  ..._int_coh.tif
#   copol:            ..._copol_coh.tif
int_files   = [f for f in all_coh_files if f.name.endswith("int_coh.tif")]
copol_files = [f for f in all_coh_files if f.name.endswith("copol_coh.tif")]

print(f"  Interferometric files : {len(int_files)}")
print(f"  Copol files           : {len(copol_files)}")

# ---------------------------------------------------------------
# Random sampling – draw up to SAMPLE_SIZE from each list
# ---------------------------------------------------------------
rng = random.Random(RANDOM_SEED)

int_sample   = rng.sample(int_files,   min(SAMPLE_SIZE, len(int_files)))
copol_sample = rng.sample(copol_files, min(SAMPLE_SIZE, len(copol_files)))

print(f"\nSampled {len(int_sample)} interferometric and "
      f"{len(copol_sample)} copol files for QA/QC.")

## 3. Data Validation

The `validate_coherence` function opens each file with `rioxarray` and checks:

| Check | Criterion |
|---|---|
| Value range | All valid (non-NaN) pixels lie **strictly within [0, 1]** |
| NaN / NoData fraction | Reported as a percentage |

A **PASS** is printed when both the minimum and maximum values are within
[0, 1]; a **FAIL** is printed otherwise.

In [ ]:
def validate_coherence(file_path: Path) -> bool:
    """
    Validate a single coherence GeoTIFF.

    Opens *file_path* with rioxarray and checks that all valid (non-NaN)
    pixels fall within the physically meaningful coherence range [0, 1].
    Prints a human-readable summary and returns True on pass, False on fail.

    Parameters
    ----------
    file_path : Path
        Path to a coherence GeoTIFF.

    Returns
    -------
    bool
        True if validation passes, False otherwise.
    """
    # Open with rioxarray; squeeze out the band dimension so we work with a
    # 2-D (y, x) array.  masked=True replaces NoData values with NaN.
    da = xr.open_dataarray(file_path, engine="rasterio", masked=True).squeeze()

    # Extract the underlying numpy array and work with NaN-safe functions
    arr = da.values

    total_pixels = arr.size
    nan_pixels   = int(np.sum(np.isnan(arr)))
    valid_pixels = total_pixels - nan_pixels

    nan_pct = (nan_pixels / total_pixels * 100) if total_pixels > 0 else float("nan")

    if valid_pixels > 0:
        # Use nanmin/nanmax so any remaining NaNs are safely ignored
        vmin  = float(np.nanmin(arr))
        vmax  = float(np.nanmax(arr))
        vmean = float(np.nanmean(arr))
    else:
        vmin = vmax = vmean = float("nan")

    # Coherence γ must satisfy 0 ≤ γ ≤ 1
    passed = (not np.isnan(vmin)) and (vmin >= 0.0) and (vmax <= 1.0)
    status = "PASS ✓" if passed else "FAIL ✗"

    print(
        f"[{status}]  {file_path.name}\n"
        f"         Min={vmin:.4f}  Max={vmax:.4f}  Mean={vmean:.4f}  "
        f"NaN={nan_pct:.1f}%  Valid pixels={valid_pixels:,}\n"
    )

    da.close()
    return passed

In [ ]:
# ------------------------------------------------------------------
# Run validation on the sampled interferometric coherence files
# ------------------------------------------------------------------
print("=" * 60)
print("INTERFEROMETRIC COHERENCE – VALIDATION")
print("=" * 60)

int_results = [validate_coherence(f) for f in int_sample]

n_pass = sum(int_results)
print(f"Result: {n_pass} / {len(int_results)} files passed.\n")

In [ ]:
# ------------------------------------------------------------------
# Run validation on the sampled copol coherence files
# ------------------------------------------------------------------
print("=" * 60)
print("COPOL COHERENCE – VALIDATION")
print("=" * 60)

copol_results = [validate_coherence(f) for f in copol_sample]

n_pass = sum(copol_results)
print(f"Result: {n_pass} / {len(copol_results)} files passed.")

## 4. Downsampled Preview Plots

Loading full-resolution GeoTIFFs into a notebook can crash the kernel on large
scenes.  `plot_coherence_sample` therefore **downsamples each raster
immediately after loading** (10× in both spatial dimensions via
`.coarsen(...).mean()`, or plain array slicing as a fallback) before handing
it to Matplotlib.

In [ ]:
import re

# Regex that matches the output filename convention produced by calc_coherence.py
# Interferometric: {site}_{flight}_{date1}_{date2}_{pol}_{segment}_int_coh.tif
# Copol:           {site}_{flight}_{date}_HV-VH_{segment}_copol_coh.tif
_TITLE_RE = re.compile(
    r"^(?P<site>[^_]+)"
    r"_(?P<flight>\d+)"
    r"_(?P<dates>[\w-]+_[\w-]+)"
    r"_(?P<pol>[A-Z]{2}(?:-[A-Z]{2})?)"
    r"_(?P<segment>s\d+)"
)


def _make_title(file_path: Path) -> str:
    """Extract metadata from the filename and return a concise subplot title."""
    m = _TITLE_RE.match(file_path.name)
    if m:
        d = m.groupdict()
        return (
            f"{d['site']} | flight {d['flight']}\n"
            f"dates: {d['dates']} | pol: {d['pol']} | seg: {d['segment']}"
        )
    # Fallback: use the bare filename if the pattern doesn't match
    return file_path.name


def plot_coherence_sample(
    file_paths: list,
    title_prefix: str = "",
    coarsen_factor: int = 10,
    ncols: int = 3,
) -> None:
    """
    Plot a downsampled preview of each coherence GeoTIFF in *file_paths*.

    Each file is opened, **immediately downsampled** by *coarsen_factor* in
    both spatial dimensions, then drawn into a subplot grid.  NaN values are
    rendered as transparent (``set_bad`` on the colormap) so they do not
    interfere with the colour scale.

    Parameters
    ----------
    file_paths : list of Path
        Coherence GeoTIFFs to plot.
    title_prefix : str, optional
        Short prefix prepended to the overall figure title.
    coarsen_factor : int, optional
        Spatial downsampling factor applied in each dimension (default 10).
    ncols : int, optional
        Number of subplot columns (default 3).
    """
    if not file_paths:
        print("No files to plot.")
        return

    nrows = int(np.ceil(len(file_paths) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 5, nrows * 4),
        squeeze=False,
    )

    # Colormap with NaN rendered as light grey so invalid pixels are visible
    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="lightgrey")

    for idx, fpath in enumerate(file_paths):
        ax = axes[idx // ncols][idx % ncols]

        # -----------------------------------------------------------------
        # Load & downsample
        # Prefer xarray coarsen so coordinate information is preserved; fall
        # back to plain numpy slicing if the array is too small to coarsen.
        # -----------------------------------------------------------------
        da = (
            xr.open_dataarray(fpath, engine="rasterio", masked=True)
            .squeeze()  # drop the band dimension
        )

        # Attempt coarsen; if the dimension is smaller than coarsen_factor
        # xarray will raise, so we fall back to slicing.
        try:
            da_ds = (
                da
                .coarsen(x=coarsen_factor, y=coarsen_factor, boundary="trim")
                .mean()
            )
        except Exception:
            # Fallback: plain numpy slicing (no coordinate update needed for display)
            da_ds = da.isel(
                x=slice(None, None, coarsen_factor),
                y=slice(None, None, coarsen_factor),
            )

        # Extract values; NaNs from masked raster are preserved as-is
        arr = da_ds.values

        # -----------------------------------------------------------------
        # Plot
        # -----------------------------------------------------------------
        im = ax.imshow(
            arr,
            vmin=0, vmax=1,
            cmap=cmap,
            origin="upper",
            interpolation="nearest",
        )
        ax.set_title(_make_title(fpath), fontsize=8)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="coherence")

        da.close()

    # Hide any unused subplot axes
    for idx in range(len(file_paths), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        f"{title_prefix} coherence – downsampled ×{coarsen_factor} preview",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# Plot the sampled interferometric coherence files
# ------------------------------------------------------------------
plot_coherence_sample(int_sample, title_prefix="Interferometric")

In [ ]:
# ------------------------------------------------------------------
# Plot the sampled copol coherence files
# ------------------------------------------------------------------
plot_coherence_sample(copol_sample, title_prefix="Copol")